In [1]:
import json
from pathlib import Path
from tqdm.notebook import tqdm
import pprint

In [2]:
p=pprint.PrettyPrinter()

In [3]:
Root=Path("/home/abhishek/Desktop/projects/RAG_MOLE/data/raw/ToolBench/data/data/answer")

g1_files=list((Root/"G1_answer").glob("*.json"))
g2_files=list((Root/"G2_answer").glob("*.json"))
g3_files=list((Root/"G3_answer").glob("*.json"))

In [4]:
def count_successful_finishes(node):

    count = 0

    if (
        node["node_type"] == "Action"
        and node["description"] == "Finish"
    ):

        if len(node["children"]) > 0:

            child = node["children"][0]

            if (
                child["node_type"] == "Action Input"
                and child.get("observation_code") == 3
            ):
                count += 1

    for child in node["children"]:
        count += count_successful_finishes(child)

    return count

In [5]:
def subtree_has_success(node):

    if (
        node["node_type"] == "Action"
        and node["description"] == "Finish"
    ):

        if len(node["children"]) > 0:

            child = node["children"][0]

            if (
                child["node_type"] == "Action Input"
                and child.get("observation_code") == 3
            ):
                return True

    for child in node["children"]:

        if subtree_has_success(child):
            return True

    return False

In [6]:
def extract_successful_trace(root):

    trace = []
    final_answer = None

    def dfs(node):

        nonlocal final_answer

        # ---------------------------------------
        # Successful Finish
        # ---------------------------------------

        if (
            node["node_type"] == "Action"
            and node["description"] == "Finish"
        ):

            if len(node["children"]) == 0:
                return False

            child = node["children"][0]

            if (
                child["node_type"] == "Action Input"
                and child.get("observation_code") == 3
            ):

                try:

                    finish_json = json.loads(child["description"])

                    final_answer = finish_json.get(
                        "final_answer",
                        child["description"]
                    )

                except:

                    final_answer = child["description"]

                return True


        # ---------------------------------------
        # Explore children
        # ---------------------------------------

        for child in node["children"]:

            if dfs(child):

                if node["node_type"] == "Action":

                    action_input = node["children"][0]

                    trace.append({

                        "step": 0,          # temporary

                        "api_name": node["description"],

                        "arguments_raw":
                            action_input.get("description"),

                        "response_raw":
                            action_input.get("observation"),

                        "observation_code":
                            action_input.get("observation_code")

                    })

                return True

        return False


    dfs(root)

    trace.reverse()

    for i, step in enumerate(trace, start=1):

        step["step"] = i

    return {

        "trace": trace,

        "final_answer": final_answer

    }

In [11]:
def process_file(file):

    with open(file, "r", encoding="utf-8") as f:

        sample = json.load(f)

    root = sample["tree"]["tree"]

    # Skip non-canonical trajectories
    if count_successful_finishes(root) != 1:

        return None

    result = extract_successful_trace(root)

    return {

        "query": sample["answer_generation"]["query"],

        "trace": result["trace"],

        "final_answer": result["final_answer"]

    }

In [12]:
sample = process_file(g1_files[0])
p.pprint(sample)

{'final_answer': 'I apologize, but I am currently unable to retrieve the list '
                 "of valid zip codes starting with '646' or the details of the "
                 "zip code '64630'. Please try again later.",
 'query': "I'm a real estate agent looking for properties in Mexico. Can you "
          "help me by giving a list of valid zip codes that start with '646'? "
          "Additionally, please provide the details of the zip code '64630', "
          'including the municipality, state, and the list of colonies.',
 'trace': [{'api_name': 'search_zip_codes_for_mexico_zip_codes',
            'arguments_raw': '{\n  "codigo_postal": "646"\n}',
            'observation_code': 12,
            'response_raw': '{"error": "request invalid, data error. '
                            'status_code=503", "response": ""}',
            'step': 1},
           {'api_name': 'zip_code_information_for_mexico_zip_codes',
            'arguments_raw': '{\n  "zip_code": "64630"\n}',
            

In [13]:
def process_dataset(files, output_folder):

    output_folder.mkdir(parents=True, exist_ok=True)

    kept = 0
    skipped = 0

    for file in tqdm(files):

        result = process_file(file)

        if result is None:

            skipped += 1
            continue

        kept += 1

        output_file = output_folder / file.name

        with open(output_file, "w", encoding="utf-8") as f:

            json.dump(
                result,
                f,
                indent=4,
                ensure_ascii=False
            )

    print(f"Kept    : {kept}")
    print(f"Skipped : {skipped}")

In [14]:
OUTPUT_DIR = Path("/home/abhishek/Desktop/projects/RAG_MOLE/data/intermediate")

(OUTPUT_DIR / "G1").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "G2").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "G3").mkdir(parents=True, exist_ok=True)

In [15]:
process_dataset(g1_files, OUTPUT_DIR / "G1")
process_dataset(g2_files, OUTPUT_DIR / "G2")
process_dataset(g3_files, OUTPUT_DIR / "G3")

  0%|          | 0/72783 [00:00<?, ?it/s]

Kept    : 47453
Skipped : 25330


  0%|          | 0/29417 [00:00<?, ?it/s]

Kept    : 21928
Skipped : 7489


  0%|          | 0/24286 [00:00<?, ?it/s]

Kept    : 14380
Skipped : 9906
